# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, following the Croissant schema specification.

### Dataset Source

The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a FrozenBox, not a dict

print(f"{getattr(metadata, 'name', '(no name found)')}: {getattr(metadata, 'description', '(no description found)')}")


## 2. Data Overview
Review record sets and their fields, referencing each by their `@id`.

> We'll print all available record sets, the fields within each, and all column and field `@id`s so that all code further below references entities using the `@id`.

In [ ]:
# List record sets and their field/column ids from the Croissant metadata
record_sets = list(dataset.record_sets.keys())

print("Record sets found in this dataset:")
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"- RecordSet @id: {rs_id}  [name: {getattr(rs, 'name', '-')}]\n  Fields:")
    for fld_id, field in rs.fields.items():
        print(f"    - Field @id: {fld_id}   [name: {getattr(field, 'name', '-')}, type: {getattr(field, 'data_type', '-')}]" )
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> Use the record set and field `@id`s from the overview above. We will extract all record sets into DataFrames.

In [ ]:
# Load all records for each available record set into pandas DataFrames
dataframes = dict()

for rs_id in record_sets:
    rs_records = list(dataset.records(record_set=rs_id))  # This yields dicts per record
    df = pd.DataFrame(rs_records)
    dataframes[rs_id] = df
    print(f"RecordSet @id: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")

# If there is only one record set, pick that for analysis, else let user select.
if len(record_sets) == 1:
    main_rs_id = record_sets[0]
else:
    main_rs_id = record_sets[0]  # Or manually pick.
print(f"\nMain record set chosen for demo: {main_rs_id}\n")

# Show first few rows of the main record set
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common processing steps: filtering, normalizing, and grouping. All columns/fields are accessed by their `@id`s as listed above.

- We will:
    - Filter records by a chosen numeric field (e.g., age, if available)
    - Normalize that field
    - Group by a categorical field (e.g., sex, anatomical site)

If your dataset has different variables, use the `@id`s from the data overview above.

In [ ]:
# Automatically select a numeric field and a group field for demonstration.
df = dataframes[main_rs_id]

numeric_field_id = None
group_field_id = None
for column in df.columns:
    # Try to detect numeric columns by type or by @id/name logic
    if df[column].dtype.kind in 'ifc' and numeric_field_id is None:
        numeric_field_id = column
    # Try to detect sex/gender/location as grouping field by column name heuristics
    if any(k in column.lower() for k in ['sex', 'gender', 'site', 'location', 'group']) and group_field_id is None:
        group_field_id = column

if numeric_field_id is None:
    # Fallback: pick the first field that looks numeric after converting
    for column in df.columns:
        try:
            df[column] = pd.to_numeric(df[column], errors='coerce')
            if df[column].notna().sum() > 0:
                numeric_field_id = column
                break
        except Exception:
            continue

print(f"Numeric field chosen (@id): {numeric_field_id}")
print(f"Group field chosen (@id): {group_field_id}")

# Proceed only if these fields are found
if numeric_field_id and group_field_id:
    threshold = df[numeric_field_id].mean()  # Simple example: threshold = mean
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.1f} (mean):")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped (mean) {numeric_field_id} by {group_field_id}:")
    print(grouped_df)
else:
    print("Not enough usable fields detected for EDA (numeric or group).")

## 5. Visualization

Visualize the distribution of the chosen numeric field, grouped by the group field, if both are available.

All axes and legends reference the schema `@id`s. Adjust fields as necessary for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field grouped by group field
if numeric_field_id and group_field_id:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Distribution of {numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("Insufficient fields for visualization.")

## 6. Conclusion

- This notebook loaded and explored the FAIR² dataset defined by a Croissant schema, referencing all entities by their `@id`.
- We dynamically loaded all available record sets, reviewed their schema, performed basic filtering, normalization, grouping, and plotted key distributions.
- For a production workflow, update filtering/grouping fields by inspecting the precise `@id` and description from the data overview above, following FAIR and Croissant best practices for reproducible ML data pipelines.
